In [1]:
# Google Colab / local setup
import os, sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CODE_ROOT = Path("/content/drive/MyDrive/msc project/code")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
        str(CODE_ROOT), 'catboost==1.2.10', 'rapidfuzz==3.14.3',
    ])
else:
    CODE_ROOT = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").exists()
    )
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
CODE = CODE_ROOT

# A — Regression-channel trading

This separate BTC experiment tests whether hourly regression channels support profitable five-minute reversal entries on development data through June 2025.

The three-layer rule uses only completed observations at the decision time.

| Layer | Definition |
|---|---|
| Macro side | Daily close versus its 200-day average |
| Channel | 60-hour rolling log-price OLS with 10th/90th residual bands |
| Entry | Five-minute reversal near the eligible channel edge |

Channel detection defines eligible structures, not guaranteed fills or returns.

Resting limit entries, swing-based stops and completed-leg targets are fixed when the order is placed; rising-channel LONG and falling-channel SHORT samples are evaluated separately.

In [2]:
from IPython.display import Markdown
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(CODE_ROOT))
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

from features.linear_channels import compute_linear_regression_channels, label_channel_regime

DATA = CODE_ROOT / "data"
CACHE = CODE_ROOT / "experiments" / "cache" / "channel_study"
DEV_END = pd.Timestamp("2025-07-01", tz="UTC")

SPLITS = pd.DataFrame(
    [("dev", "2021-01-01", "2025-07-01", "rules, geometry, thresholds"),
     ("tune", "2025-07-01", "2025-10-01", "not reached; branch not promoted"),
     ("forward", "2025-10-01", "2026-04-01", "not evaluated in this branch"),
     ("lockbox", "2026-04-01", "2026-07-01", "not evaluated in this branch")],
    columns=["stage", "from", "to", "used for"])
display(Markdown('Method: The registered chronology separates rule development, calibration, forward and Q2 eligibility.'))
display(SPLITS)
display(Markdown('The channel branch stops before later-stage promotion.'))

h1 = pd.read_parquet(DATA / "btcusdt_1h_2021_2026.parquet")
m5 = pd.read_parquet(DATA / "btcusdt_5min_2021_2026.parquet")

Method: The registered chronology separates rule development, calibration, forward and Q2 eligibility.

,stage,from,to,used for
0,dev,2021-01-01,2025-07-01,"rules, geometry, thresholds"
1,tune,2025-07-01,2025-10-01,not reached; branch not promoted
2,forward,2025-10-01,2026-04-01,not evaluated in this branch
3,lockbox,2026-04-01,2026-07-01,not evaluated in this branch


The channel branch stops before later-stage promotion.

## 1. How much structure the market offers

The channel is the unit the datasets are built around, so its count sets the scale
of everything downstream. Bars inside one channel share a regime and are not
independent draws; the channel count, not the bar count, bounds what can be claimed.

In [3]:
from IPython.display import Markdown
# How many channels exist. Trades and episodes have been reported elsewhere; this
# counts the structures themselves, which is what the two datasets are built around.
hi = h1[h1.index < DEV_END]
rows = []
for window in (60, 90, 120):
    ch = compute_linear_regression_channels(hi, window=window, log_price=True,
                                            method="quantile", quantile=0.10)
    ch["channel_slope"] = ch["channel_slope"] * 1e4          # bps per 1h bar
    reg = label_channel_regime(ch, min_slope=5.0, min_r2=0.40)
    run = (reg != reg.shift()).cumsum()
    entry = {"window": window}
    for kind in ("up", "down"):
        sizes = run[reg == kind].value_counts()
        entry[f"{kind} channels"] = len(sizes)
        entry[f"{kind} mean hours"] = round(sizes.mean(), 1) if len(sizes) else np.nan
    entry["total"] = entry["up channels"] + entry["down channels"]
    entry["share of time"] = round(float((reg != "none").mean()), 3)
    rows.append(entry)

census = pd.DataFrame(rows)
days = (DEV_END - pd.Timestamp("2021-01-01", tz="UTC")).days
census["per month"] = (census["total"] / (days / 30.4)).round(1)
display(Markdown('Method: Completed regression channels are counted before applying trade-entry conditions.'))
display(census)
display(Markdown('Channel structures are more numerous than executed trades.'))

Method: Completed regression channels are counted before applying trade-entry conditions.

,window,up channels,up mean hours,down channels,down mean hours,total,share of time,per month
0,60,234,34.3,215,32.4,449,0.380,8.3
1,90,145,53.0,132,46.3,277,0.350,5.1
2,120,104,66.9,84,64.3,188,0.314,3.5


Channel structures are more numerous than executed trades.

## 2. Design decisions, each measured

Every completed run writes its configuration next to its result. The rows below
differ only in the decisions under test — the target rule, the position capacity,
and the geometry thresholds — so the comparison is like for like.

Two readings matter more than the ranking. A target placed on the far channel edge
is reached in a small minority of trades, because at a deep entry it sits five or
more risk units away; a target sized to the completed leg is nearer and is reached
far more often. And a capacity limit is a portfolio choice rather than a signal one:
holding the signal fixed, it removes trades without improving the ones that remain.

In [4]:
from IPython.display import Markdown
# Every completed run, read from its own artefact. The columns that differ between
# rows are the design decisions under test; everything else is held fixed.
rows = []
for d in sorted(CACHE.glob("*/")):
    cfg_p, sum_p = d / "config.json", d / "summary.json"
    if not (cfg_p.exists() and sum_p.exists()):
        continue
    c, s = json.loads(cfg_p.read_text()), json.loads(sum_p.read_text())
    rows.append({
        "target": c.get("target_mode"),
        "capacity": c.get("max_concurrent") or "unlimited",
        "min R2": c.get("min_r2"),
        "risk bps": f'{c.get("min_risk_bps"):g}-{c.get("max_risk_bps"):g}',
        "min R:R": c.get("min_rr"),
        "hold": c.get("max_hold_bars"),
        "trades": s.get("num_trades"),
        "per day": round(s.get("trades_per_day", np.nan), 2),
        "TP-first": s.get("tp_first_rate"),
        "gross R": s.get("mean_r_gross"),
        "net R": s.get("mean_r_net"),
    })
runs = pd.DataFrame(rows).sort_values("trades").reset_index(drop=True)
for col in ("TP-first", "gross R", "net R"):
    runs[col] = runs[col].astype(float).round(4)
display(Markdown('Method: Registered geometry and execution variants share the same development window and cost treatment.'))
display(runs)
display(Markdown('None of the tested mechanical configurations clears costs.'))

best = runs.loc[runs["net R"].idxmax()]

Method: Registered geometry and execution variants share the same development window and cost treatment.

,target,capacity,min R2,risk bps,min R:R,hold,trades,per day,TP-first,gross R,net R
0,rail,unlimited,0.6,40-250,1.5,96,8,0.00,0.0000,-0.1160,-0.2097
1,rail,1,0.6,40-250,1.5,96,13,0.01,0.0000,-0.2891,-0.4241
2,measured,unlimited,0.4,40-250,1.5,288,1166,0.71,0.2727,0.0329,-0.0645
3,measured,unlimited,0.4,40-250,1.2,288,1766,1.08,0.3216,0.0427,-0.0500
4,measured,unlimited,0.2,40-250,1.2,288,2121,1.29,0.3305,0.0535,-0.0376
5,rail,3,0.6,25-250,0.0,96,2135,1.30,0.0684,0.0012,-0.1355


None of the tested mechanical configurations clears costs.

## 3. From signal to filled trade

The funnel shows where candidates are lost. Losses split into two kinds that must
not be conflated: geometry rejections are a statement about the trade, while
capacity and fill failures are statements about execution.

In [5]:
from IPython.display import Markdown
# The deployed configuration, from signal to filled trade.
# The deployed configuration, not the largest run: sections 2 and 3 must describe
# the same thing, and a superseded variant can easily hold more rows.
def _deployed(d):
    c = json.loads((d / "config.json").read_text())
    s = json.loads((d / "summary.json").read_text())
    return (c.get("target_mode") == "measured", c.get("max_concurrent") is None,
            s.get("mean_r_net", -9))

run_dir = max((d for d in CACHE.glob("*/")
               if (d / "summary.json").exists() and (d / "config.json").exists()),
              key=_deployed)
s = json.loads((run_dir / "summary.json").read_text())
sk = s["skipped"]
signals = s["num_trades"] + sum(sk.values())

funnel = pd.DataFrame(
    [("reversal signals", signals, 1.0)]
    + [(f"lost to {k}", -v, -v / signals) for k, v in sk.items() if v]
    + [("filled trades", s["num_trades"], s["num_trades"] / signals)],
    columns=["stage", "count", "share of signals"])
funnel["share of signals"] = funnel["share of signals"].round(3)
display(Markdown('Method: The selected configuration counts successive geometry, signal and fill filters.'))
display(funnel)
display(Markdown('Geometry eligibility and unfilled orders explain different losses of coverage.'))
cfg = json.loads((run_dir / "config.json").read_text())

ev = pd.read_parquet(run_dir / "events.parquet")

Method: The selected configuration counts successive geometry, signal and fill filters.

,stage,count,share of signals
0,reversal signals,7480,1.000
1,lost to geometry,-3994,-0.534
2,lost to unfilled,-1329,-0.178
3,lost to channel_cancelled,-36,-0.005
4,filled trades,2121,0.284


Geometry eligibility and unfilled orders explain different losses of coverage.

## 4. How far the frequency can be pushed

Every constraint that could be loosened to admit more trades is swept below: the two
geometry floors, the resting limit's price offset, and how long that limit is left
in the book. The signal itself is untouched throughout, so each row differs from the
reference only in what it is willing to accept or wait for.

A single relaxation improves both count and per-trade result; the rest buy volume by
admitting worse candidates. That asymmetry is the finding, and it places a ceiling on
frequency for any configuration that retains an edge.

In [6]:
from IPython.display import Markdown
# Every lever that could add trades, each measured through the same runner. Count and
# edge are shown together, because while the strategy is loss-making a larger count is
# a larger loss, not a better result: the metric that matters here is net R per trade.
csv = CACHE / "frequency_levers_dev.csv"
levers = pd.read_csv(csv).sort_values("net R", ascending=False).reset_index(drop=True)
display(Markdown('Method: One admission constraint is varied at a time using the same replay engine.'))
display(levers)
display(Markdown('Higher admission frequency does not create a positive net edge.'))

ref = levers[levers["variant"].str.contains("reference")].iloc[0]
best = levers.iloc[0]

Method: One admission constraint is varied at a time using the same replay engine.

,variant,trades,per day,lost geom,unfilled,TP%,gross R,se,net R,total R
0,min_rr 1.2,1766,1.08,3593,1208,0.322,0.0427,0.0338,-0.0500,-88.3
1,fill window 8 bars,1855,1.13,3880,818,0.322,0.0410,0.0328,-0.0522,-96.8
2,fill window 12 bars,1884,1.15,4005,650,0.321,0.0338,0.0325,-0.0598,-112.7
3,min_risk 30 bps,2333,1.42,3026,1208,0.322,0.0501,0.0303,-0.0623,-145.3
4,limit offset 0 bps,2009,1.22,4551,20,0.320,0.0308,0.0311,-0.0629,-126.3
5,"risk>=40, rr>=1.5, offset 5 (reference)",1166,0.71,4193,1208,0.273,0.0329,0.0455,-0.0645,-75.3
6,min_rr 0,2771,1.69,2588,1208,0.359,0.0161,0.0244,-0.0707,-195.8
7,min_rr 1,2758,1.68,2601,1208,0.358,0.0146,0.0245,-0.0723,-199.3
8,limit offset 2 bps,1882,1.15,4071,622,0.315,0.0160,0.0323,-0.0774,-145.7
9,offset 2 + fill window 8,1918,1.17,4238,412,0.314,0.0139,0.0319,-0.0798,-153.0


Higher admission frequency does not create a positive net edge.

## 5. Is the candidate pool separable?

A ranking model can only extract structure that already varies across candidates.
Each stratum below is defined by a quantity known at the decision bar.

In [7]:
from IPython.display import Markdown
# Whether the pool a ranking model would receive is separable at all. A filter can
# only extract what varies; if every stratum returns the same edge, more candidates
# is only more commission.
filled = ev[ev["filled"].astype(bool)].copy()
out = []
for col, bins, labels in [
    ("channel_confluence", [-0.1, 0.5, 1.1], ["one window", "two or more"]),
    ("episode_trade_number", [-0.1, 0.5, 2.5, 99], ["first", "2nd-3rd", "4th+"]),
    ("risk_bps_decision", None, None),
    ("rr_planned_decision", None, None),
]:
    if col not in filled:
        continue
    if bins is None:
        q = filled[col].quantile([0, 1/3, 2/3, 1]).to_numpy()
        grp = pd.cut(filled[col], bins=np.unique(q), include_lowest=True)
    else:
        grp = pd.cut(filled[col], bins=bins, labels=labels)
    g = filled.groupby(grp, observed=True)["r_net"]
    for name, sub in g:
        if len(sub) < 30:
            continue
        out.append({"feature": col, "stratum": str(name), "n": len(sub),
                    "mean net R": round(sub.mean(), 3),
                    "se": round(sub.std(ddof=1) / np.sqrt(len(sub)), 3)})
display(Markdown('Method: Candidate strata use quantities available when the order is placed, with identical execution scoring.'))
display(pd.DataFrame(out))
display(Markdown('Candidate separation must be demonstrated before adding a ranking model.'))

Method: Candidate strata use quantities available when the order is placed, with identical execution scoring.

,feature,stratum,n,mean net R,se
0,channel_confluence,one window,633,-0.249,0.054
1,channel_confluence,two or more,2030,-0.044,0.033
2,episode_trade_number,2nd-3rd,149,-0.224,0.128
3,episode_trade_number,4th+,2472,-0.083,0.029
4,risk_bps_decision,"(40.002, 54.371]",888,-0.164,0.053
5,risk_bps_decision,"(54.371, 78.088]",887,-0.034,0.050
6,risk_bps_decision,"(78.088, 248.613]",888,-0.080,0.044
7,rr_planned_decision,"(1.2000000000000002, 1.52]",888,-0.017,0.040
8,rr_planned_decision,"(1.52, 2.181]",887,-0.118,0.046
9,rr_planned_decision,"(2.181, 8.335]",888,-0.143,0.060


Candidate separation must be demonstrated before adding a ranking model.

## 6. Can a model rank what the rule produces?

The rule does not clear its costs, so the question becomes whether its candidates
differ from one another in a way that is readable before the trade is placed. The
control is take-every-candidate: a model that cannot beat it has not found a weak
signal, it has found a homogeneous pool.

In [8]:
from IPython.display import Markdown
# The ranking test. Scores come from forward-chaining folds cut between channel
# episodes, so no candidate is scored by a model that saw its own channel, and the
# threshold is swept over out-of-fold predictions rather than fitted ones.
run = max((d for d in CACHE.glob("*/") if (d / "ranking_e8_table.csv").exists()),
          key=lambda d: (d / "ranking_e8_table.csv").stat().st_mtime)
table = pd.read_csv(run / "ranking_e8_table.csv")
display(Markdown('Method: Forward-chaining folds are split between channel episodes before ranking candidates.'))
display(table)
display(Markdown('Ranking does not establish a cost-covering strategy.'))

control = table.loc[table["selection"] == "take all", "mean_net_r"].iloc[0]
best = table.loc[table["selection"] != "take all", "mean_net_r"].max()

Method: Forward-chaining folds are split between channel episodes before ranking candidates.

,selection,n,share,mean_net_r,ci_low,ci_high,episodes,beats_control
0,take all,1885,1.0,-0.090800,-0.183506,0.004230,171,False
1,top 10% by model,189,0.1,-0.082199,-0.279626,0.112408,50,False
2,top 20% by model,377,0.2,0.011890,-0.126368,0.147208,90,False
3,top 30% by model,566,0.3,0.041661,-0.080450,0.156798,107,True
4,top 50% by model,943,0.5,-0.009974,-0.117810,0.093584,141,False


Ranking does not establish a cost-covering strategy.

### 6.1 One model, or one per side?

The design keeps rising-channel longs and falling-channel shorts apart. That is the
cleaner arrangement, and it is affordable only if each side carries enough channels
to train on.

Two readings decide it. If a pooled model merely preferred whichever side performs
better, its selected slices would be skewed towards that side and it would not rank
within either. The within-side rows below test exactly that.

In [9]:
from IPython.display import Markdown
# Whether one model over both sides beats one model per side, and whether a pooled
# score ranks inside each side or merely prefers the better-performing one.
sides = pd.read_csv(run / "ranking_e8_sides.csv")
display(Markdown('Method: LONG and SHORT rankings are evaluated separately on held-out channel episodes.'))
display((sides).fillna('Not estimable'))
display(Markdown('Side-specific results remain insufficient for promotion.'))

def lift(model, scope):
    sub = sides[(sides["model"] == model) & (sides["scope"] == scope)]
    if len(sub) < 2:
        return np.nan
    top = sub[sub["slice"].str.contains("top 30")]["mean_net_r"].iloc[0]
    allr = sub[sub["slice"].str.contains("all")]["mean_net_r"].iloc[0]
    return round(top - allr, 4)

comp = pd.DataFrame([
    {"side": "long", "pooled model": lift("pooled", "within long"),
     "own model": lift("long-only", "long")},
    {"side": "short", "pooled model": lift("pooled", "within short"),
     "own model": lift("short-only", "short")},
])
display(Markdown('Method: A pooled ranking model is compared with separate-side models on the same candidates.'))
display(comp)
display(Markdown("Pooling does not resolve the branch's economic deficit."))

sl = sides[(sides["model"] == "pooled") & (sides["scope"] == "both sides")]

Method: LONG and SHORT rankings are evaluated separately on held-out channel episodes.

,model,scope,slice,n,short share,mean_net_r
0,pooled,both sides,top 10%,189,0.46,-0.0822
1,pooled,both sides,top 30%,566,0.461,0.0417
2,pooled,both sides,all,1885,0.497,-0.0908
3,pooled,within long,top 30%,285,Not estimable,0.0525
4,pooled,within long,all,948,Not estimable,-0.1166
5,long-only,long,take all,939,Not estimable,-0.1184
6,long-only,long,top 30% by model,282,Not estimable,-0.0017
7,pooled,within short,top 30%,281,Not estimable,0.0053
8,pooled,within short,all,937,Not estimable,-0.0647
9,short-only,short,take all,856,Not estimable,-0.0104


Side-specific results remain insufficient for promotion.

Method: A pooled ranking model is compared with separate-side models on the same candidates.

,side,pooled model,own model
0,long,0.1691,0.1167
1,short,0.0700,-0.0771


Pooling does not resolve the branch's economic deficit.

## Conclusion and limitations

No tested mechanical configuration clears costs, so the branch is not promoted.
The sample contains roughly 400 channels and one bear market; uncertainty must therefore be clustered by episode, not individual trade.
About 160 design configurations create multiple-testing risk, and resting-limit fill rates are modelled rather than directly observed.